# Week 3 – Topic 3 Lab: Parts A & B (PDF/TXT pathway)
**Project:** AI-Powered Contract and Terms & Conditions Analyzer

Run the cells **in order**. Source files are the lecturer's *synthetic practice dataset* (not real contracts), so nothing here measures clause-detection accuracy.

| Cell | Lab step |
|---|---|
| 1, 2, 3 | A2 – workspace, optional ZIP, verify uploads |
| 4, 4A | B2 – source selection + lineage helpers |
| A3 | A3 – inspect two files |
| 5A1, 5A2 | B2/B3 – Pathway A: clean, compare chunking, export |
| B3 | B3 – output validation |
| B4.1, 6A, 6B | B4 – lineage registration, manifest, verification |

## A2 – Workspace setup

In [ ]:
# Cell 1 — Workspace setup (RUN FIRST)
from pathlib import Path

BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Compatibility aliases
RAW = RAW_DIR
PROCESSED = PROCESSED_DIR
BASE = BASE_DIR
OUT = PROCESSED_DIR

print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)

In [ ]:
# Cell 2 — Extract practice ZIP (OPTIONAL: only if you uploaded Topic_3_Dummy_Datasets.zip into data/raw)
from zipfile import ZipFile

zip_path = RAW_DIR / "Topic_3_Dummy_Datasets.zip"
if zip_path.is_file():
    with ZipFile(zip_path) as zf:
        zf.extractall(RAW_DIR)
    print("Dummy dataset pack extracted.")
    print("Location:", RAW_DIR / "topic3_dummy_datasets")
else:
    print("No ZIP found.")
    print("Continue with your separately uploaded project files.")

In [ ]:
# Cell 3 — Confirm uploaded inputs
files = sorted(p for p in RAW_DIR.rglob("*") if p.is_file())
print("Input count:", len(files))
for p in files:
    print(f"- {p.relative_to(RAW_DIR)}  ({p.stat().st_size} bytes)")
assert files, "No files in RAW_DIR - upload into /content/week3_project/data/raw"
print("Upload verified. Select your relevant input(s) for Step A3.")

## B2 – Common helpers (Cell 4 and 4A)

In [ ]:
# Cell 4 — Source selection + lineage utilities
from pathlib import Path
import json, hashlib
from datetime import datetime, timezone

def select_source(relative_path=None, filename=None, suffixes=None):
    if relative_path:
        selected = RAW_DIR / relative_path
        if not selected.is_file():
            raise FileNotFoundError(f"Source not found: {selected}")
        return selected
    matches = sorted(
        p for p in RAW_DIR.rglob("*")
        if p.is_file()
        and (filename is None or p.name == filename)
        and (suffixes is None or p.suffix.lower() in suffixes)
    )
    print("Candidates:", [str(p.relative_to(RAW_DIR)) for p in matches])
    if len(matches) != 1:
        raise ValueError("Select exactly one source. Set relative_path to an exact file under RAW_DIR.")
    return matches[0]

def source_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def save_manifest(records):
    manifest = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "records": records,
    }
    target = BASE_DIR / "data" / "week3_manifest.json"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Saved lineage:", target)
    return manifest

print("Cell 4 completed successfully.")
print("Available functions: select_source(), source_hash(), save_manifest()")

In [ ]:
# Cell 4A — Common lineage registry
lineage_entries = {}

def register_lineage(source, output, processing):
    source = Path(source).resolve()
    output = Path(output).resolve()
    if not source.is_file():
        raise FileNotFoundError(f"Source file not found: {source}")
    if not output.is_file():
        raise FileNotFoundError(f"Prepared output not found: {output}")
    if output.stat().st_size == 0:
        raise ValueError(f"Prepared output is empty: {output}")
    if not isinstance(processing, str) or not processing.strip():
        raise ValueError("A processing description is required.")
    processing = processing.strip()
    if processing.upper().startswith("REPLACE"):
        raise ValueError("Replace the placeholder with the actual method.")
    source.relative_to(RAW_DIR.resolve())
    output.relative_to(PROCESSED_DIR.resolve())
    key = (str(source), str(output))
    lineage_entries[key] = {"source": source, "output": output, "processing": processing}
    print("Registered:", source.name, "->", output.name)

print("Common lineage registry ready.")
print("Registered entries:", len(lineage_entries))

## A1 – Requirement → data mapping (Topic 2)

| Requirement ID | Needed information / feature | Candidate source | Why usable / constraint |
|---|---|---|---|
| **FR-03 / FR-05** (text extraction; cleaning and segmentation) | Readable text from an uploaded PDF/TXT, split into sections with context kept | **Now:** synthetic `student_handbook_demo.pdf` + `faq_demo.txt` (practice only). **Later:** permission-checked public contracts / T&C documents | Synthetic files exercise extraction and chunking only. They contain no contract clauses, so they cannot support FR-06/07 or NFR-04 claims. |
| **NFR-06 / RAI-04** (privacy; test docs public or authorised) and **RAI-01 / NFR-05** (source grounding and traceability) | Source identity, version/owner, access rights, checksum; chunk-to-source link | Same files + manifest | Synthetic, redistributable teaching data: safe for GitHub. Real contracts must be public or authorised before they go in the repo. |

**Practice dataset notice:** source recorded as *Synthetic practice dataset*. It will be replaced by a permission-checked contract / T&C dataset (public T&C documents or authorised sample agreements) selected for later model development and evaluation.

## A3 – Select and inspect two sources

In [ ]:
# Cell A3 — Inspect two representative sources
# If you uploaded the files directly (not the ZIP), edit these two relative paths.
PDF_REL = "topic3_dummy_datasets/01_documents/student_handbook_demo.pdf"
TXT_REL = "topic3_dummy_datasets/01_documents/faq_demo.txt"

!pip -q install pypdf
from pypdf import PdfReader
import re

pdf_path = select_source(relative_path=PDF_REL)
txt_path = select_source(relative_path=TXT_REL)
SOURCES = [pdf_path, txt_path]

def read_pages(path):
    if path.suffix.lower() == ".pdf":
        reader = PdfReader(str(path))
        return [(i + 1, page.extract_text() or "") for i, page in enumerate(reader.pages)]
    if path.suffix.lower() == ".txt":
        return [(1, path.read_text(encoding="utf-8"))]
    raise ValueError(f"Pathway A needs PDF or TXT, got {path.suffix}")

for p in SOURCES:
    pages = read_pages(p)
    text = "\n".join(t for _, t in pages)
    print("=" * 60)
    print("Source:", p.relative_to(RAW_DIR))
    print("Type:", p.suffix.lower(), "| bytes:", p.stat().st_size)
    print("Pages/segments:", len(pages), "| extracted characters:", len(text))
    print("SHA-256:", source_hash(p))
    print("Content sample:", repr(text[:300]))
    assert text.strip(), "No usable text: check extraction or OCR needs."

## Pathway A – clean and compare two chunking approaches

In [ ]:
# Cell 5A1 — Clean + compare fixed-size vs structure-aware chunking
def clean_text(text):
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def fixed_chunks(s, size=160):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    return [s[i:i + size].strip() for i in range(0, len(s), size) if s[i:i + size].strip()]

def structure_chunks(s, suffix):
    if suffix == ".pdf":
        # new chunk at each "Section N:" heading; title/metadata stay together as chunk 0
        chunks, current = [], []
        for line in (x.strip() for x in s.splitlines() if x.strip()):
            if re.match(r"^Section\s+\d+:", line, re.I) and current:
                chunks.append("\n".join(current)); current = [line]
            else:
                current.append(line)
        if current:
            chunks.append("\n".join(current))
        return chunks
    # TXT: keep each blank-line separated block (title / Q&A pair) together
    return [b.strip() for b in re.split(r"\n\s*\n", s) if b.strip()]

prepared = {}
for p in SOURCES:
    text = clean_text("\n".join(t for _, t in read_pages(p)))
    fx, st = fixed_chunks(text), structure_chunks(text, p.suffix.lower())
    prepared[p.name] = {"clean": text, "fixed": fx, "structure": st}
    print("=" * 60, "\n", p.name, "| cleaned chars:", len(text))
    print("Fixed-160 chunks:", len(fx), "| Structure-aware chunks:", len(st))
    print("-- fixed sample:", [c[:70] for c in fx[:3]])
    print("-- structure sample:", [c[:70] for c in st[:3]])

In [ ]:
# B3 comparison table (observed evidence)
print(f"{'Source':28} {'Fixed-160':>10} {'Structure':>10}  Heading/Q&A split by fixed?")
for name, d in prepared.items():
    # a fixed chunk "splits" a unit if it starts or ends mid-line of a heading/Q or A line
    broken = sum(1 for c in d["fixed"] if not c.rstrip().endswith((".", ":", "?")))
    print(f"{name:28} {len(d['fixed']):>10} {len(d['structure']):>10}  {broken} fixed chunk(s) end mid-sentence")

**B3 decision (fill in after reading the output above):**
Selected approach: **structure-aware chunking**. *Benefit:* each chunk keeps a whole section heading or Q&A pair, which supports FR-05 (preserve context) and RAI-01 / NFR-05 (each chunk can be cited back to a section). *Accepted limitation:* it relies on the headings of these samples (`Section N:` / blank lines); real contracts use numbered clauses and will need rules tested on real documents. The samples are tiny, so this is a qualitative choice, not a performance result.

In [ ]:
# Cell 5A2 — Export prepared records (selected approach) + B3 validation
approach = ("structure-aware chunking: PDF split at 'Section N:' headings; "
            "TXT split at blank-line blocks; whitespace/newline cleanup only")

records = []
for p in SOURCES:
    digest = source_hash(p)
    for i, chunk in enumerate(prepared[p.name]["structure"], 1):
        records.append({
            "source_file": p.name,
            "source_sha256": digest,
            "chunk_id": f"{p.stem}-{i:03d}",
            "chunk_index": i,
            "method": "structure_aware",
            "char_count": len(chunk),
            "text": chunk,
        })

prepared_path = PROCESSED_DIR / "prepared_text.jsonl"
with prepared_path.open("w", encoding="utf-8") as out:
    for row in records:
        out.write(json.dumps(row, ensure_ascii=False) + "\n")

# B3 basic output contract
assert prepared_path.exists(), "Prepared output missing."
assert prepared_path.stat().st_size > 0, "Prepared output is empty."
assert records and all(r["text"].strip() for r in records), "Empty chunk found"
with prepared_path.open(encoding="utf-8") as fh:
    exported = [json.loads(l) for l in fh if l.strip()]
assert exported == records, "Reloaded export differs from in-memory records"
print("Output:", prepared_path, "Size:", prepared_path.stat().st_size)
print("Validation: file exists, non-empty, reloads identically.")
print("Total chunks exported:", len(records))
print("Representative records:")
for r in (records[0], records[-1]):
    print(" ", r["chunk_id"], "|", repr(r["text"][:80]))

# Repeatability check
rerun = [structure_chunks(clean_text("\n".join(t for _, t in read_pages(p))), p.suffix.lower()) for p in SOURCES]
assert rerun == [prepared[p.name]["structure"] for p in SOURCES]
print("Repeatability: second run identical.")

## B4 – Lineage registration, manifest and verification

In [ ]:
# B4.1 — Register Pathway A (one entry per source -> prepared_text.jsonl)
for p in SOURCES:
    register_lineage(source=p, output=prepared_path, processing=approach)
print("Registered entries:", len(lineage_entries))

In [ ]:
# CELL 6A — Generate common lineage manifest
assert lineage_entries, "No lineage entries registered."
lineage_records = []
for i, entry in enumerate(lineage_entries.values(), 1):
    source, output, processing = entry["source"], entry["output"], entry["processing"]
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert output.stat().st_size > 0, "Empty output"
    lineage_records.append({
        "source_id": f"S{i:02d}",
        "source_filename": str(source.relative_to(RAW_DIR.resolve())),
        "source_sha256": source_hash(source),
        "prepared_output": str(output.relative_to(BASE_DIR.resolve())),
        "prepared_sha256": source_hash(output),
        "processing_note": processing,
        "source_version": "Not verified (synthetic practice dataset)",
        "access_rights": "Lecturer-supplied synthetic teaching data; lab use only",
        "status": "Prepared; quality review pending",
    })
manifest = save_manifest(lineage_records)
print("\nCOMMON LINEAGE MANIFEST GENERATED")
print("Total lineage entries:", len(manifest["records"]))

In [ ]:
# CELL 6B — Verify common lineage manifest
manifest_path = BASE_DIR / "data" / "week3_manifest.json"
assert manifest_path.is_file(), "Common lineage manifest not found."
saved = json.loads(manifest_path.read_text(encoding="utf-8"))
saved_records = saved.get("records", [])
assert saved_records, "No lineage records found in saved manifest."
assert len(saved_records) == len(lineage_entries), "Saved lineage count does not match registration."
for rec in saved_records:
    src = RAW_DIR / rec["source_filename"]
    out = BASE_DIR / rec["prepared_output"]
    assert src.is_file(), f"Missing source: {src}"
    assert out.is_file(), f"Missing output: {out}"
    assert source_hash(src) == rec["source_sha256"], f"Source checksum mismatch: {src}"
    assert source_hash(out) == rec["prepared_sha256"], f"Prepared checksum mismatch: {out}"
print("COMMON LINEAGE VALIDATION: PASS")
print("Verified records:", len(saved_records))
print("Manifest:", manifest_path)